# اخراجات کے دعوے کا تجزیہ

یہ نوٹ بک اس بات کا مظاہرہ کرتی ہے کہ کس طرح ایسے ایجنٹ بنائے جائیں جو پلگ انز کا استعمال کرتے ہوئے مقامی رسید کی تصاویر سے سفر کے اخراجات کو پراسیس کریں، اخراجات کا دعوی ای میل تیار کریں، اور پائی چارٹ کے ذریعے اخراجات کے ڈیٹا کو بصری شکل میں پیش کریں۔ ایجنٹس کام کے سیاق و سباق کی بنیاد پر فنکشنز کا انتخاب کرتے ہیں۔

مراحل:
1. OCR ایجنٹ مقامی رسید کی تصویر کو پراسیس کرتا ہے اور سفر کے اخراجات کا ڈیٹا نکالتا ہے۔
2. ای میل ایجنٹ اخراجات کا دعوی ای میل تیار کرتا ہے۔

### سفر کے اخراجات کی ایک مثال:
فرض کریں آپ ایک ملازم ہیں جو کاروباری ملاقات کے لیے دوسرے شہر میں سفر کر رہا ہے۔ آپ کی کمپنی کی پالیسی ہے کہ تمام معقول سفر سے متعلق اخراجات کی واپسی کی جائے۔ یہاں ممکنہ سفر کے اخراجات کا خلاصہ ہے:
- ٹرانسپورٹیشن:
آپ کے گھر کے شہر سے مقام منزل تک اور واپس راؤنڈ ٹرپ کے لیے ہوائی کرایہ۔
ہوائی اڈے جانے اور آنے کے لیے ٹیکسی یا رائیڈ ہیلنگ خدمات۔
مقام منزل میں مقامی ٹرانسپورٹ (جیسے عوامی ٹرانزٹ، کرائے کی گاڑیاں، یا ٹیکسی)۔

- رہائش:
میٹنگ کے مقام کے قریب درمیانے درجے کے کاروباری ہوٹل میں تین راتوں کا قیام۔

- کھانے:
کمپنی کی روزانہ خوراک کی پالیسی کے مطابق ناشتہ، دوپہر کا کھانا، اور رات کا کھانا۔

- متفرق اخراجات:
ہوائی اڈے پر پارکنگ فیس۔
ہوٹل میں انٹرنیٹ کی فیس۔
ٹپس یا چھوٹے سروس چارجز۔

- دستاویزات:
آپ تمام رسیدیں (فلائٹس، ٹیکسی، ہوٹل، کھانے وغیرہ) اور مکمل اخراجات کی رپورٹ واپسی کے لیے جمع کرواتے ہیں۔


## مطلوبہ لائبریریاں درآمد کریں

نوٹ بک کے لیے ضروری لائبریریاں اور ماڈیول درآمد کریں۔


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## اخراجات کے ماڈلز کی تعریف کریں

 انفرادی اخراجات کے لیے پائیڈینٹک ماڈل بنائیں اور ایک ExpenseFormatter کلاس تیار کریں جو صارف کی درخواست کو منظم اخراجات کے ڈیٹا میں تبدیل کرے۔

 ہر خرچ کو درج ذیل فارمیٹ میں پیش کیا جائے گا:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## آلات کی تعریف کرنا - ای میل تخلیق کرنا

اخراجات کا دعوٰی جمع کرانے کے لیے ایک ای میل بنانے کے لیے ایک ٹول فنکشن بنائیں۔
- یہ ٹول Microsoft Agent Framework کے `@tool` ڈیکیٹر استعمال کرتا ہے۔
- یہ اخراجات کی کل رقم کا حساب لگاتا ہے اور تفصیلات کو ای میل کے متن میں ترتیب دیتا ہے۔


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# رسید کی تصاویر سے سفر کے اخراجات نکالنے کا آلہ

سفر کے اخراجات نکالنے کے لیے ایک آلے کا فنکشن بنائیں جو رسید کی تصاویر سے کام لے۔
- یہ آلہ مائیکروسافٹ ایجنٹ فریم ورک کے `@tool` ڈیکوریٹر کا استعمال کرتا ہے۔
- یہ رسید کی تصویر کو پڑھتا ہے، اسے base64 میں انکوڈ کرتا ہے، اور ایجنٹ کے تجزیہ کے لیے ڈیٹا URI واپس کرتا ہے۔


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## اخراجات کی پروسیسنگ

ایجنٹس کو متعین کریں اور انہیں `WorkflowBuilder` استعمال کرتے ہوئے ایک ترتیب وار ورک فلو میں جوڑیں۔
- OCR ایجنٹ رسید کی تصویر سے منظم شدہ خرچ کا ڈیٹا `load_receipt_image` ٹول کے ذریعے نکالتا ہے۔
- ای میل ایجنٹ نکالا گیا ڈیٹا لیتا ہے اور `generate_expense_email` ٹول کے ذریعے ایک پیشہ ورانہ خرچ کا دعویٰ ای میل تیار کرتا ہے۔
- `WorkflowBuilder` `add_edge` کے ساتھ ایک ترتیب وار پائپ لائن بناتا ہے: OCR ایجنٹ → ای میل ایجنٹ۔


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## مرکزی فعل

تسلسل میں ورک فلو تیار کریں اور اسے چالو کرکے رسید کی تصویر کو پروسیس کریں اور خرچ کے دعوے کا ای میل تیار کریں۔


> **نوٹ:** یہ ورک فلو فی الحال رسید کی تصویر کو بیس64 متن کے طور پر پاس کرتا ہے، جسے زیادہ تر چیٹ ماڈلز (جیسے gpt-5-mini) تصویر کے طور پر نہیں سمجھیں گے۔
> یہ ماڈل کے سیاق و سباق کی ونڈو سے تجاوز بھی کر سکتا ہے۔ بہتر ہے کہ Azure AI Vision (یا کوئی اور OCR ٹول) کے ساتھ OCR چلائیں اور صرف نکالا گیا متن بھیجیں، یا تصویر کو `image_url` پیغام کے طور پر بھیجنے کے لیے دوبارہ ترتیب دیں۔
> اگر آپ صرف سیاق و سباق کی غلطیوں سے بچنا چاہتے ہیں، تو ایک چھوٹی رسید کی تصویر یا زیادہ بڑی سیاق و سباق کی ونڈو والا ماڈل استعمال کرنے کی کوشش کریں۔


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
